In [15]:
import pandas as pd
import numpy as np

In [16]:
data = pd.read_csv('Energy_consumption.csv')

In [17]:
data.isna().sum()

Timestamp            0
Temperature          0
Humidity             0
SquareFootage        0
Occupancy            0
HVACUsage            0
LightingUsage        0
RenewableEnergy      0
DayOfWeek            0
Holiday              0
EnergyConsumption    0
dtype: int64

In [18]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Timestamp          1000 non-null   object 
 1   Temperature        1000 non-null   float64
 2   Humidity           1000 non-null   float64
 3   SquareFootage      1000 non-null   float64
 4   Occupancy          1000 non-null   int64  
 5   HVACUsage          1000 non-null   object 
 6   LightingUsage      1000 non-null   object 
 7   RenewableEnergy    1000 non-null   float64
 8   DayOfWeek          1000 non-null   object 
 9   Holiday            1000 non-null   object 
 10  EnergyConsumption  1000 non-null   float64
dtypes: float64(5), int64(1), object(5)
memory usage: 86.1+ KB


In [19]:
print(data.columns)

Index(['Timestamp', 'Temperature', 'Humidity', 'SquareFootage', 'Occupancy',
       'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday',
       'EnergyConsumption'],
      dtype='object')


In [20]:
data.head()

,Timestamp,Temperature,Humidity,SquareFootage,Occupancy,HVACUsage,LightingUsage,RenewableEnergy,DayOfWeek,Holiday,EnergyConsumption
0,2022-01-01 00:00:00,25.139433,43.431581,1565.693999,5,On,Off,2.774699,Monday,No,75.364373
1,2022-01-01 01:00:00,27.731651,54.225919,1411.064918,1,On,On,21.831384,Saturday,No,83.401855
2,2022-01-01 02:00:00,28.704277,58.907658,1755.715009,2,Off,Off,6.764672,Sunday,No,78.270888
3,2022-01-01 03:00:00,20.080469,50.371637,1452.316318,1,Off,On,8.623447,Wednesday,No,56.519850
4,2022-01-01 04:00:00,23.097359,51.401421,1094.130359,9,On,Off,3.071969,Friday,No,70.811732


In [21]:
data["HVACUsage"] = data["HVACUsage"].apply(lambda x: 1 if x == "ON" else 0)

In [22]:
data["LightingUsage"] = data["LightingUsage"].apply(lambda x: 1 if x == "ON" else 0)

In [23]:
data["Holiday"] = data["Holiday"].apply(lambda x: 1 if x == "Yes" else 0)

In [24]:
data["DayOfWeek"].unique()

array(['Monday', 'Saturday', 'Sunday', 'Wednesday', 'Friday', 'Thursday',
       'Tuesday'], dtype=object)

In [25]:
from sklearn.preprocessing import OrdinalEncoder

In [26]:
order= ['Saturday', 'Sunday', 'Monday','Tuesday', 'Wednesday', 'Thursday', 'Friday']
DayOfWeek_encoder = OrdinalEncoder(categories=[order], dtype=int)
data["DayOfWeek"] = DayOfWeek_encoder.fit_transform(data[["DayOfWeek"]])
data["DayOfWeek"].head()


0    2
1    0
2    1
3    4
4    6
Name: DayOfWeek, dtype: int32

In [27]:
import pickle

with open("encoder.pkl", "wb") as f:
    pickle.dump(DayOfWeek_encoder, f)


In [28]:
data["Timestamp"] = pd.to_datetime(data["Timestamp"])

data["Hour"] = data["Timestamp"].dt.hour
data["Weekday"] = data["Timestamp"].dt.weekday   # 0=Mon ... 6=Sun
data["Month"] = data["Timestamp"].dt.month
data["IsWeekend"] = data["Weekday"].isin([5, 6]).astype(int)
data.drop("Timestamp", axis=1, inplace=True)


In [29]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Temperature        1000 non-null   float64
 1   Humidity           1000 non-null   float64
 2   SquareFootage      1000 non-null   float64
 3   Occupancy          1000 non-null   int64  
 4   HVACUsage          1000 non-null   int64  
 5   LightingUsage      1000 non-null   int64  
 6   RenewableEnergy    1000 non-null   float64
 7   DayOfWeek          1000 non-null   int32  
 8   Holiday            1000 non-null   int64  
 9   EnergyConsumption  1000 non-null   float64
 10  Hour               1000 non-null   int32  
 11  Weekday            1000 non-null   int32  
 12  Month              1000 non-null   int32  
 13  IsWeekend          1000 non-null   int32  
dtypes: float64(5), int32(5), int64(4)
memory usage: 90.0 KB


In [30]:
data.to_csv(r"C:\Users\Dell\Desktop\broadway\cleaned_energy_consumption.csv", index=False)

trainning

In [31]:
from sklearn.linear_model import LinearRegression

In [32]:
X = data.drop(columns="EnergyConsumption")
y = data[["EnergyConsumption"]]

In [33]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


In [34]:
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [35]:
from sklearn.metrics import mean_absolute_error

test_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, test_pred)
print("MAE:", mae)


MAE: 4.58431635691735


In [36]:
model.coef_ 

array([[ 1.98093409e+00, -6.48213690e-02, -4.83252352e-04,
         4.89255162e-01,  0.00000000e+00, -6.38378239e-16,
         9.85017557e-02, -3.72486682e-02,  8.00020230e-01,
        -4.17560425e-02, -1.21328932e-03,  6.42095128e-01,
         7.06734193e-01]])

In [37]:
model.intercept_ 

array([26.73805668])

In [38]:
import pickle
with open ("energy_consumption.pkl", "wb") as file:
    pickle.dump(model,file)

In [39]:
print("Trainning Completed & Model Saved!")

Trainning Completed & Model Saved!


In [40]:
#interfencing

In [41]:
import pickle
with open("energy_consumption.pkl", "rb") as file:
    my_model = pickle.load(file)   

In [42]:
with open("encoder.pkl", "rb") as file:
    DayOfWeek_encoder = pickle.load(file)

In [43]:
test_data = pd.read_csv("test_data.csv")
test_data.head()

,Temperature,Humidity,SquareFootage,Occupancy,HVACUsage,LightingUsage,RenewableEnergy,DayOfWeek,Holiday,Hour,Weekday,Month,IsWeekend
0,26.135291,48.498729,1303.715986,8,0,0,5.667656,5,1,17,5,1,1
1,27.185351,33.581353,1457.566456,4,0,0,4.312261,4,0,17,0,1,0
2,25.154024,37.593489,1869.598548,6,0,0,5.176600,2,1,20,0,1,0
3,29.938483,57.730876,1596.822747,9,0,0,28.751923,2,0,12,4,1,0
4,22.296357,40.901362,1568.750108,8,0,0,21.578186,3,0,3,1,1,0


In [44]:
test_pred = my_model.predict(test_data)

In [45]:
test_data["EnergyConsumption"] = test_pred

In [48]:
test_data.to_csv("predicted_energy.csv", index=False)
test_data.head()

,Temperature,Humidity,SquareFootage,Occupancy,HVACUsage,LightingUsage,RenewableEnergy,DayOfWeek,Holiday,Hour,Weekday,Month,IsWeekend,EnergyConsumption
0,26.135291,48.498729,1303.715986,8,0,0,5.667656,5,1,17,5,1,1,80.455570
1,27.185351,33.581353,1457.566456,4,0,0,4.312261,4,0,17,0,1,0,79.874317
2,25.154024,37.593489,1869.598548,6,0,0,5.176600,2,1,20,0,1,0,77.204103
3,29.938483,57.730876,1596.822747,9,0,0,28.751923,2,0,12,4,1,0,88.827439
4,22.296357,40.901362,1568.750108,8,0,0,21.578186,3,0,3,1,1,0,73.939683


In [ ]:
from sklearn.metrics import mean_absolute_error

test_pred = my_model.predict(X_test)
mae_test = mean_absolute_error(y_test, test_pred)
print(f"MAE on Test Data: {mae_test:.2f}")

MAE on Test Data: 4.58


In [47]:
print(X.columns)
print(len(X.columns))


Index(['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage',
       'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'Hour',
       'Weekday', 'Month', 'IsWeekend'],
      dtype='object')
13
